# NYC Taxi Big Data Pipeline: 2015 vs 2025
**Project Goal:** Process and analyze over 195 million rows of NYC Taxi trip records using Apache Spark to identify market trends, clean anomalies, and evaluate the impact of ride-sharing apps over a decade.

In [1]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

# 1. Initialize Spark Session with optimized memory for Big Data processing (8GB RAM)
print("Initializing Spark Cluster...")
spark = SparkSession.builder \
    .appName("NYC_Taxi_Big_Data_Analysis") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print(f"Spark version: {spark.version} is ready for action")

Initializing Spark Cluster...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/21 12:24:16 WARN Utils: Your hostname, Kamils-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.116 instead (on interface en0)
26/08/21 12:24:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/21 12:24:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.4 is ready for action


## 1. Data Ingestion
Loading raw Parquet files (highly compressed columnar storage) for the years 2015 and 2025, alongside a CSV lookup table for Taxi Zones.

In [2]:
# Load modern data (2025)
print("Loading 2025 Parquet files...")
df_2025 = spark.read.parquet("data/2025/*.parquet")

# Load historical data (2015)
print("Loading 2015 Parquet files...")
df_2015 = spark.read.parquet("data/2015/*.parquet")

# Load Taxi Zone lookup table (CSV)
print("Loading Taxi Zone Dictionary...")
df_zones = spark.read.csv("lookup/*.csv", header=True, inferSchema=True)

print(f"Total rows to process: {df_2025.count() + df_2015.count():,}")

Loading 2025 Parquet files...


26/08/21 12:24:24 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/2025/*.parquet.
java.io.FileNotFoundException: File data/2025/*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSo

Loading 2015 Parquet files...
Loading Taxi Zone Dictionary...


26/08/21 12:24:25 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/2015/*.parquet.
java.io.FileNotFoundException: File data/2015/*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSo

Total rows to process: 194,761,833


## 2. Data Cleaning & Feature Engineering (2025 Dataset)
Calculating trip duration from timestamps, filtering out system errors (e.g., negative fares, zero-minute trips), and joining the dimension table to resolve location IDs.

In [4]:
# Calculate duration in minutes using safe Unix Timestamp conversion
df_2025_cleaned = df_2025.withColumn(
    "duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

# Filter out anomalies and dirty data
df_2025_cleaned = df_2025_cleaned.filter(
    (F.col("duration_min") > 0) &           # Trip must be longer than 0 minutes
    (F.col("fare_amount") > 0) &            # Fare must be positive
    (F.col("PULocationID").between(1, 263)) # Valid NYC Zone IDs only
)

# Join with the CSV lookup table to get human-readable Borough and Zone names
df_joined = df_2025_cleaned.join(
    df_zones,
    df_2025_cleaned["PULocationID"] == df_zones["LocationID"],
    "inner" 
)

# Rename columns for better readability
df_joined = df_joined.withColumnRenamed("Borough", "Pickup_Borough") \
                     .withColumnRenamed("Zone", "Pickup_Zone")

print("Cleaned and Joined Dataset Sample")
df_joined.select("tpep_pickup_datetime", "fare_amount", "duration_min", "Pickup_Borough").show(5)

Cleaned and Joined Dataset Sample
+--------------------+-----------+------------------+--------------+
|tpep_pickup_datetime|fare_amount|      duration_min|Pickup_Borough|
+--------------------+-----------+------------------+--------------+
| 2025-05-01 00:07:06|       18.4|             17.15|     Manhattan|
| 2025-05-01 00:07:44|        8.6| 6.716666666666667|     Manhattan|
| 2025-05-01 00:15:56|       10.0|              7.95|     Manhattan|
| 2025-05-01 00:00:09|       40.8|25.333333333333332|        Queens|
| 2025-05-01 00:45:07|       10.0| 7.633333333333334|     Manhattan|
+--------------------+-----------+------------------+--------------+
only showing top 5 rows


## 3. Business Aggregations (2025 Market Overview)
Grouping over 45 million cleaned records by Borough to calculate total trips, average tips, and average fares.

In [5]:
# Generate aggregations
df_business_report = df_joined.groupBy("Pickup_Borough").agg(
    F.count("*").alias("Total_Trips"),
    F.round(F.avg("tip_amount"), 2).alias("Avg_Tip_USD"),
    F.round(F.avg("fare_amount"), 2).alias("Avg_Fare_USD"),
    F.round(F.avg("duration_min"), 2).alias("Avg_Duration_Minutes")
).orderBy(F.col("Total_Trips").desc())

print("2025 Market Report by Borough")
df_business_report.show()

2025 Market Report by Borough


[Stage 15:===================================================>    (11 + 1) / 12]

+--------------+-----------+-----------+------------+--------------------+
|Pickup_Borough|Total_Trips|Avg_Tip_USD|Avg_Fare_USD|Avg_Duration_Minutes|
+--------------+-----------+-----------+------------+--------------------+
|     Manhattan|   38991016|       2.64|       16.55|                15.1|
|        Queens|    4300716|       7.44|       49.94|                35.0|
|      Brooklyn|    1551141|       0.56|       26.99|               27.46|
|         Bronx|     354798|       0.17|        28.7|               31.42|
|           EWR|       5877|       10.3|       89.18|                2.59|
| Staten Island|       3981|       1.23|       36.05|               26.05|
+--------------+-----------+-----------+------------+--------------------+



## 4. The Decade Comparison: 2015 vs 2025
Combining nearly 200 million rows to analyze the drastic shift in the NYC Taxi market over 10 years.

In [6]:
# Clean 2015 dataset
df_2015_cleaned = df_2015.withColumn(
    "duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
).filter(
    (F.col("duration_min") > 0) & 
    (F.col("fare_amount") > 0)
)

# Aggregate 2015 stats
df_2015_report = df_2015_cleaned.agg(
    F.lit("2015").alias("Year"),
    F.count("*").alias("Total_Trips"),
    F.round(F.avg("tip_amount"), 2).alias("Avg_Tip_USD"),
    F.round(F.avg("fare_amount"), 2).alias("Avg_Fare_USD"),
    F.round(F.avg("duration_min"), 2).alias("Avg_Duration_Min")
)

# Aggregate 2025 stats
df_2025_report = df_2025_cleaned.agg(
    F.lit("2025").alias("Year"),
    F.count("*").alias("Total_Trips"),
    F.round(F.avg("tip_amount"), 2).alias("Avg_Tip_USD"),
    F.round(F.avg("fare_amount"), 2).alias("Avg_Fare_USD"),
    F.round(F.avg("duration_min"), 2).alias("Avg_Duration_Min")
)

# UNION the two datasets
df_decade_comparison = df_2015_report.union(df_2025_report).orderBy("Year")

print("The Decline of Yellow Cabs (10-Year Trend)")
df_decade_comparison.show()

The Decline of Yellow Cabs (10-Year Trend)


[Stage 19:====================================================>   (15 + 1) / 16]

+----+-----------+-----------+------------+----------------+
|Year|Total_Trips|Avg_Tip_USD|Avg_Fare_USD|Avg_Duration_Min|
+----+-----------+-----------+------------+----------------+
|2015|  145783665|       1.73|       12.94|            20.8|
|2025|   45207529|       3.01|       20.19|           17.55|
+----+-----------+-----------+------------+----------------+



## 5. Exporting Results
Saving the finalized, cleaned 2025 dataset back to the optimized Parquet format, and exporting the business summary to CSV for stakeholders.

In [7]:
# Write the massive cleaned dataset to Parquet
print("Writing cleaned 2025 data to Parquet...")
df_joined.write.mode("overwrite").parquet("output_data_2025_parquet")

# Write the small summary report to a single CSV file
print("Writing Business Report to CSV...")
df_business_report.coalesce(1).write.mode("overwrite").csv("business_report_csv", header=True)

print("Pipeline execution completed successfully!")

Writing cleaned 2025 data to Parquet...


Writing Business Report to CSV...


[Stage 33:===================================================>    (11 + 1) / 12]

Pipeline execution completed successfully!
